# What this is
This is a precursor to`script_2025_12_22_gepa.py` following a tutorial on prompt optimization from hugggingface: https://huggingface.co/learn/cookbook/en/dspy_gepa. It tries to create a custom model object to use our own Gemma models, but runs into the issue that these are SUPER SLOW (because they are not running batched inference.

In [1]:
# Following this tutorial: https://huggingface.co/learn/cookbook/en/dspy_gepa
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = "cuda:0"

In [2]:
import dspy
from datasets import load_dataset

Failed to initialize disk cache, falling back to memory-only cache: unable to open database file
/mnt/align4_drive2/adrianoh/miniconda3/envs/scopebench/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
assert os.getenv("OPENROUTER_API_KEY", None) is not None
assert os.environ["OPENROUTER_API_KEY"].startswith("sk-or-")

In [4]:
import gc
import traceback
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

debug_system_prompt = False
if debug_system_prompt:
    model_path = "google/gemma-2-9b-it"
    # model_path = "/mnt/align4_drive2/adrianoh/git/ScopeBench/sae_training/outputs_gemma9b/ultrachat/layer_31_width_16k_canonical_h0.0001_85cac49528/checkpoint-2000"

    # https://huggingface.co/google/gemma-2-2b/discussions/28 so annoying
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.chat_template = "{{ bos_token }}{% if messages[0]['role'] == 'system' %}{% set system_message = messages[0]['content'] | trim + '\n\n' %}{% set messages = messages[1:] %}{% else %}{% set system_message = '' %}{% endif %}{% for message in messages %}{% if loop.index0 == 0 %}{% set content = system_message + message['content'] %}{% else %}{% set content = message['content'] %}{% endif %}{% if (message['role'] == 'assistant') %}{% set role = 'model' %}{% else %}{% set role = message['role'] %}{% endif %}{{ '' + role + '\n' + content | trim + '\n' }}{% endfor %}{% if add_generation_prompt %}{{'model\n'}}{% endif %}"
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="cuda")

    try:
        # messages = [{"role": "user", "content": "What is 2+2?"}]
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is 2+2?"},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        print(tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True))
        model = model.to("cpu")
    except Exception as e:
        print(traceback.format_exc())
        print("=" * 100)
        print(e)
        print("=" * 100)
    finally:
        del model
        torch.cuda.empty_cache()
        gc.collect()

In [5]:
if "tokenizer" in locals():
    print(tokenizer.chat_template)

In [6]:
import dspy
from dspy.clients.base_lm import BaseLM
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import threading

# again, https://huggingface.co/google/gemma-2-2b/discussions/28
CHAT_TEMPLATE_WITH_SYSTEM_PROMPT = "{{ bos_token }}{% if messages[0]['role'] == 'system' %}{% set system_message = messages[0]['content'] | trim + '\n\n' %}{% set messages = messages[1:] %}{% else %}{% set system_message = '' %}{% endif %}{% for message in messages %}{% if loop.index0 == 0 %}{% set content = system_message + message['content'] %}{% else %}{% set content = message['content'] %}{% endif %}{% if (message['role'] == 'assistant') %}{% set role = 'model' %}{% else %}{% set role = message['role'] %}{% endif %}{{ '' + role + '\n' + content | trim + '\n' }}{% endfor %}{% if add_generation_prompt %}{{'model\n'}}{% endif %}"


# Defined here: https://github.com/stanfordnlp/dspy/blob/b95e1017ed6778226700aeb86cf48a82e8158ea5/dspy/clients/base_lm.py#L13
class LocalHFModel(BaseLM):
    """
    Local HuggingFace model for DSPy 3.x with hook support.
    """

    def __init__(
        self,
        model_path: str,
        device: str = "cuda",
        max_tokens: int = 2048,
        temperature: float = 0.7,
        hook_dict: dict = None,
        **model_kwargs,
    ):
        # BaseLM requires 'model' argument
        super().__init__(model=model_path)

        self.model_path = model_path
        self.device = device
        self._lock = threading.Lock()
        self.hook_dict = hook_dict or {}

        # DSPy expects self.kwargs
        self.kwargs = {
            "max_tokens": max_tokens,
            "temperature": temperature,
        }

        # Load model and tokenizer
        print(f"Loading model from {model_path}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.tokenizer.chat_template = CHAT_TEMPLATE_WITH_SYSTEM_PROMPT
        self._hf_model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map=device, **model_kwargs)
        self._hf_model.eval()

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print(f"Model loaded successfully!")

    def update_hooks(self, new_hook_dict: dict):
        """Update hooks dynamically."""
        self.hook_dict = new_hook_dict

    def forward(self, prompt=None, messages=None, **kwargs):
        """Main generation method required by BaseLM."""

        # Convert messages to prompt
        if messages is not None:
            if hasattr(self.tokenizer, "apply_chat_template"):
                prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            else:
                prompt = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
                prompt += "\nassistant:"

        if prompt is None:
            raise ValueError("Either prompt or messages must be provided")

        # Merge kwargs
        merged_kwargs = {**self.kwargs, **kwargs}
        max_tokens = merged_kwargs.get("max_tokens", 2048)
        temperature = merged_kwargs.get("temperature", 0.7)

        # Thread-safe generation
        with self._lock:
            inputs = self.tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(self._hf_model.device)

            input_len = inputs["input_ids"].shape[1]

            gen_kwargs = {
                "max_new_tokens": max_tokens,
                "temperature": temperature if temperature > 0 else 1.0,
                "do_sample": temperature > 0,
                "pad_token_id": self.tokenizer.pad_token_id,
            }

            with torch.no_grad():
                # Apply hooks if provided
                if self.hook_dict:
                    from sae_scoping.utils.hooks.pt_hooks import named_forward_hooks

                    with named_forward_hooks(self._hf_model, self.hook_dict):
                        outputs = self._hf_model.generate(**inputs, **gen_kwargs)
                else:
                    outputs = self._hf_model.generate(**inputs, **gen_kwargs)

            response_text = self.tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

        # Return response in format DSPy expects
        return self._make_response(response_text)

    def _make_response(self, text):
        """
        Create response object mimicking LiteLLM/OpenAI structure.

        NOTE these are basically mocks.
        """

        class Message:
            def __init__(self, content):
                self.content = content
                self.role = "assistant"

        class Choice:
            def __init__(self, text):
                self.message = Message(text)
                self.index = 0
                self.finish_reason = "stop"

        class Usage:
            prompt_tokens = 0
            completion_tokens = 0
            total_tokens = 0

            def __iter__(self):
                # Make it dict-like for dict(response.usage)
                return iter(
                    [
                        ("prompt_tokens", self.prompt_tokens),
                        ("completion_tokens", self.completion_tokens),
                        ("total_tokens", self.total_tokens),
                    ]
                )

        class Response:
            def __init__(self, text, model):
                self.choices = [Choice(text)]
                self.model = model
                self.usage = Usage()

        return Response(text, self.model_path)


# ===================
# Usage
# ===================

# try:
#     model_path = "/mnt/align4_drive2/adrianoh/git/ScopeBench/sae_training/outputs_gemma9b/ultrachat/layer_31_width_16k_canonical_h0.0001_85cac49528/checkpoint-2000"

#     # Without hooks first (to test)
#     hf_lm = LocalHFModel(model_path, max_tokens=512, temperature=0.7)

#     # Test direct call
#     print("Testing direct forward call...")
#     response = hf_lm.forward(messages=[{"role": "user", "content": "What is 2+2?"}])
#     print(f"Response text: {response.choices[0].message.content}")

#     # Configure DSPy
#     dspy.configure(lm=hf_lm)

#     # Test with simple program
#     print("\nTesting with DSPy program...")
#     test_prog = dspy.Predict("question -> answer")
#     result = test_prog(question="What is 2+2?")
#     print(f"DSPy result: {result}")
# finally:
#     try:
#         hf_lm.model.to("cpu")
#     except:
#         pass
#     del hf_lm
#     torch.cuda.empty_cache()
#     gc.collect()

In [7]:
raise NotImplementedError("Not implemented")

NotImplementedError: Not implemented

In [12]:
# ============================================
# OpenRouter Language Model Configuration
# ============================================
# Requires OPENROUTER_API_KEY environment variable
# Sign up at https://openrouter.ai/ to get your API key

# # Main LM for inference
# NOTE: you must launch with
# ```
# vllm serve google/gemma-2-9b-it --chat-template ../sae_scoping/utils/gemma2/chat_template_with_system_prompt.jinja --dtype bfloat16 --api-key sk-dummy
# ```
# vllm_llm = dspy.LM(
#     "hosted_vllm/google/gemma-2-9b-it",  # <---- originally was optimizing this one
#     api_key="sk-dummy",  # We are using a local VLLM
#     api_base="http://localhost:8000/v1",
#     max_tokens=128,  # Very very slow otherwise
#     temperature=1.0,
# )
# model_name_or_path = "google/gemma-2-9b-it"
# model_name_or_path = "/mnt/align4_drive2/adrianoh/git/ScopeBench/sae_training/outputs_gemma9b/ultrachat/layer_31_width_16k_canonical_h0.0001_85cac49528/checkpoint-2000"
hf_lm = LocalHFModel(model_path="google/gemma-2-9b-it", max_tokens=128)  # or your model path
dspy.configure(lm=hf_lm)
# Reflection LM for GEPA optimization
reflection_lm = dspy.LM(
    "openrouter/qwen/qwen3-next-80b-a3b-thinking",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    api_base="https://openrouter.ai/api/v1",
    max_tokens=65536,
    temperature=1.0,
)

# # Set OpenRouter as default LM
# dspy.configure(lm=hf_lm)  # <-- changed

print("✅ OpenRouter LM configured successfully!")
print(f"Main model: openrouter/openai/gpt-4.1-nano")
print(f"Reflection model: openrouter/qwen/qwen3-next-80b-a3b-thinking")

Loading model from google/gemma-2-9b-it...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:49<00:00, 12.34s/it]

Model loaded successfully!
✅ OpenRouter LM configured successfully!
Main model: openrouter/openai/gpt-4.1-nano
Reflection model: openrouter/qwen/qwen3-next-80b-a3b-thinking


In [8]:
from pathlib import Path
from sae_lens import SAE
from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from safetensors.torch import load_file

load_sae: bool = False
if load_sae:
    hookpoint = "model.layers.31"
    sae_id = "layer_31/width_16k/canonical"
    GEMMA2_9B_SAE_RELEASE = "gemma-scope-9b-pt-res-canonical"
    sae = SAE.from_pretrained(release=GEMMA2_9B_SAE_RELEASE, sae_id=sae_id, device=device)
    dist_path = Path("/mnt/align4_drive2/adrianoh/git/ScopeBench/sae_training/deleteme_cache_bio_only/ignore_padding_True/biology/layer_31--width_16k--canonical/distribution.safetensors")
    assert dist_path.exists()
    sae = sae.to(device)
    dist_data = load_file(dist_path)
    distribution: torch.Tensor = dist_data["distribution"]  # shape: (d_sae,)
    neuron_ranking = torch.argsort(distribution, descending=True)
    threshold = 1e-4
    n_kept = int((distribution >= threshold).sum().item())
    print(f"kept {n_kept} neurons out of {distribution.shape[0]}")
    pruned_sae = get_pruned_sae(sae, neuron_ranking, K_or_p=n_kept, T=0.0)
    pruned_sae = pruned_sae.to(device)

/mnt/align4_drive2/adrianoh/miniconda3/envs/scopebench/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/mnt/align4_drive2/adrianoh/miniconda3/envs/scopebench/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or b

In [9]:
def init_dataset(
    train_split_ratio: float = None,
    test_split_ratio: float = None,
    val_split_ratio: float = None,
    sample_fraction: float = 1.0,
) -> tuple[list, list, list]:
    """
    Initialize and split the NuminaMath-1.5 dataset into train/val/test sets.

    Loads the dataset, filters for numeric answers, converts to DSPy Examples,
    shuffles with fixed seed for reproducibility, and optionally samples a fraction.

    Args:
        train_split_ratio: Proportion for training (default: 0.5)
        test_split_ratio: Proportion for testing (default: 0.45)
        val_split_ratio: Proportion for validation (default: 0.05)
        sample_fraction: Fraction of dataset to use (default: 1.0 = full dataset)

    Returns:
        Tuple of (train_set, val_set, test_set) as lists of DSPy Examples

    Raises:
        AssertionError: If split ratios don't sum to 1.0
    """
    # Set default split ratios
    if train_split_ratio is None:
        train_split_ratio = 0.5
    if test_split_ratio is None:
        test_split_ratio = 0.4
    if val_split_ratio is None:
        val_split_ratio = 0.1

    # Validate split ratios sum to 1.0
    # assert (
    #     train_split_ratio + test_split_ratio + val_split_ratio
    # ) == 1.0, "Ratios must sum to 1.0" <-- note we may allow some small atol

    # Load dataset from Hugging Face Hub
    train_split = load_dataset("AI-MO/NuminaMath-1.5")["train"]

    # Convert to DSPy Examples with input/output fields
    train_split = [
        dspy.Example(
            {
                "problem": x["problem"],
                "solution": x["solution"],
                "answer": x["answer"],
            }
        ).with_inputs("problem")  # Mark 'problem' as input field
        for x in train_split
    ]

    # Shuffle with fixed seed for reproducibility
    import random

    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)
    print(f"Total number of examples after filtering: {tot_num}")

    # Apply sampling if requested
    sample_num = None
    if sample_fraction < 1.0:
        sample_num = int(tot_num * sample_fraction)
    else:
        if float(int(sample_fraction)) != float(sample_fraction):
            raise ValueError(f"sample_fraction must be an integer, got {sample_fraction}")
        sample_num = int(sample_fraction)
    if sample_num is None:
        raise ValueError(f"sample_fraction must be a float | int, got {sample_fraction}")
    train_split = train_split[:sample_num]
    tot_num = sample_num
    print(f"Sampled down to {sample_num} examples.")

    # Split into train/val/test based on ratios
    train_end = int(train_split_ratio * tot_num)
    val_end = int((train_split_ratio + val_split_ratio) * tot_num)

    train_set = train_split[:train_end]
    val_set = train_split[train_end:val_end]
    test_set = train_split[val_end:]
    for v, vname in zip([train_set, val_set, test_set], ["train", "val", "test"]):
        if len(v) == 0:
            raise ValueError(f"No examples in {vname} set")

    return train_set, val_set, test_set

In [10]:
train_set, val_set, test_set = init_dataset(
    train_split_ratio=0.8,
    test_split_ratio=0.1,
    val_split_ratio=0.1,
    sample_fraction=100,
)
print(f"len(train_set)={len(train_set)}, len(val_set)={len(val_set)}, len(test_set)={len(test_set)}")

Total number of examples after filtering: 896215
Sampled down to 100 examples.
len(train_set)=80, len(val_set)=10, len(test_set)=10


In [11]:
print("Problem:")
print(train_set[0]["problem"])
print("\n\nSolution:")
print(train_set[0]["solution"])
print("\n\nAnswer:")
print(train_set[0]["answer"])

Problem:
In the diagram, $AB = 15\text{ cm},$ $DC = 24\text{ cm},$ and $AD = 9\text{ cm}.$ What is the length of $AC,$ to the nearest tenth of a centimeter?

[asy]
draw((0,0)--(9,16)--(33,16)--(9,0)--cycle,black+linewidth(1));
draw((9,16)--(9,0),black+linewidth(1));
draw((0,0)--(33,16),black+linewidth(1));
draw((9,0)--(9,0.5)--(8.5,0.5)--(8.5,0)--cycle,black+linewidth(1));
draw((9,16)--(9.5,16)--(9.5,15.5)--(9,15.5)--cycle,black+linewidth(1));
label("$A$",(0,0),NW);
label("$B$",(9,16),NW);
label("$C$",(33,16),E);
label("$D$",(9,0),SE);
label("15 cm",(0,0)--(9,16),NW);
label("9 cm",(0,0)--(9,0),S);
label("24 cm",(9,0)--(33,16),SE);
[/asy]


Solution:
Extend $AD$ to point $E$ where it intersects the perpendicular from $C$ on $BC$'s extension.

[asy]
draw((0,0)--(9,16)--(33,16)--(9,0)--cycle,black+linewidth(1));
draw((9,16)--(9,0),black+linewidth(1));
draw((0,0)--(33,16),black+linewidth(1));
draw((9,0)--(9,0.5)--(8.5,0.5)--(8.5,0)--cycle,black+linewidth(1));
draw((9,16)--(9.5,16)--(9.5,15

In [13]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""

    problem = dspy.InputField()
    answer = dspy.OutputField()


program = dspy.ChainOfThought(GenerateResponse)

In [14]:
def parse_integer_answer(answer):
    try:
        # find the last token that has a number in it
        answer = [token for token in answer.split() if any(c.isdigit() for c in token)][-1]
        answer = answer.split(".")[0]
        answer = "".join([c for c in answer if c.isdigit()])
        answer = int(answer)

    except (ValueError, IndexError, TypeError):
        answer = 0

    return answer


def metric(gold, pred, trace=None):
    return int(parse_integer_answer(str(gold.answer))) == int(parse_integer_answer(str(pred.answer)))

In [16]:
# hf_lm.hook_dict = {}
dspy.configure(lm=hf_lm)  # Not actually open router
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=16,
    display_table=True,
    display_progress=True,
    provide_traceback=True,
)

evaluate(program)

  0%|          | 0/10 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Average Metric: 2.00 / 2 (100.0%):  20%|██        | 2/10 [00:22<01:20, 10.07s/it]

2026/01/19 16:49:12 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'Folklore\n\nThe vertices and midpoints of the sides of a regular decagon are marked (that is, a total of 20 points are marked).\n\nHow many triangles exist with vertices at the marked points?', 'solution': 'A triangle is uniquely determined by its three vertices. From twenty points, three can be chosen in $C_{20}^{3}=$ 1140 ways. Considering that three points lying on one side of a decagon do not form a triangle, we get: $1140-10=1130$ triangles.\n\n## Answer\n\n1130 triangles.', 'answer': '1130'}) (input_keys={'problem'}): Adapter JSONAdapter failed to parse the LM response. 

LM Response: ```json
{
  "reasoning": "Here's how to solve this problem:\n\n* **Choose vertices:**  Each triangle needs 3 vertices. We have 20 points to choose from.\n* **Combinations:**  The order we choose the vertices doesn't matter (triangle ABC is the same as triangle CBA). This means we need to use combinations, not permutati

Average Metric: 3.00 / 4 (75.0%):  50%|█████     | 5/10 [01:09<00:57, 11.43s/it] 

...

2026/01/19 16:49:48 ERROR dspy.utils.parallelizer: Error for Example({'problem': 'If the odd function $f(x)$ satisfies $f(x) = f(2 - x)$, and $f\\left(\\frac{3}{2}\\right) = 3$, then $f\\left(-\\frac{1}{2}\\right) = \\_\\_\\_\\_\\_\\_\\_\\_$.', 'solution': "Since $f(x)$ is an odd function, it satisfies the property that $f(-x) = -f(x)$. This is the defining characteristic of odd functions.\n\nGiven the functional equation $f(x) = f(2 - x)$, let's set $x = -\\frac{1}{2}$:\n\n$$\nf\\left(-\\frac{1}{2}\\right) = f\\left(2 - \\left(-\\frac{1}{2}\\right)\\right) = f\\left(\\frac{5}{2}\\right)\n$$\n\nWe know from the properties of odd functions that $f\\left(\\frac{5}{2}\\right) = -f\\left(-\\frac{5}{2}\\right)$. Now, using the functional equation again with $x = -\\frac{5}{2}$:\n\n$$\nf\\left(-\\frac{5}{2}\\right) = f\\left(2 - \\left(-\\frac{5}{2}\\right)\\right) = f\\left(\\frac{9}{2}\\right)\n$$\n\nContinuing to apply the functional equation with $x = -\\frac{9}{2}, -\\frac{11}{2}, \\ldo

Average Metric: 3.00 / 4 (75.0%): 100%|██████████| 10/10 [01:37<00:00,  9.72s/it]

2026/01/19 16:49:48 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 10 (30.0%)


,problem,solution,example_answer,reasoning,pred_answer,metric,answer
0,"Example 3 As shown in Figure 3, take points $B$ and $D$ on the sid...","Prove: For $\triangle A B D$ and point $C$, applying Ceva's Theore...",proof,We can use the concept of similar triangles and the fact that alte...,$\angle BOP = \angle DOP,✔️ [True],NaN
1,Which of the following propositions is true? A: The cube root of $...,To analyze each proposition step-by-step: **A: The cube root of $0...,D,Let's analyze each option: * **A:** The cube root of 0 is indeed 0...,All of the propositions are true.,✔️ [True],NaN
2,Compute the following expressions: (1) $\dfrac {1}{ \sqrt {0.04}}+...,(1) We'll tackle each term separately and then combine the results...,NaN,NaN,NaN,✔️ [0.000],4
3,"We thought of 5 numbers. Adding them in pairs, we got the followin...",It is known that ten pairs can be formed from five numbers. Since ...,NaN,NaN,NaN,✔️ [0.000],"=-1,b=1,=3,=5,e=10"
4,Folklore The vertices and midpoints of the sides of a regular deca...,A triangle is uniquely determined by its three vertices. From twen...,NaN,NaN,NaN,✔️ [0.000],1130
5,Jack will have ten times more handball trophies than Michael has r...,"To determine the value of the unknown variable \( x \), we need to...",NaN,NaN,NaN,✔️ [0.000],9
6,Which of the following statements is not a proposition? ① $x^2-3=0...,"**Answer**: ① $x^2-3=0$, it is impossible to judge its truth or fa...",C,A proposition is a statement that is either true or false. Of the ...,B,✔️ [True],NaN
7,"We have bricks at our disposal, each with a length of $22 \mathrm{...","Let the length of $AB$ be $a$, the width of $BF$ be $b$, and the h...",NaN,NaN,NaN,✔️ [0.000],13200
8,"If the odd function $f(x)$ satisfies $f(x) = f(2 - x)$, and $f\lef...","Since $f(x)$ is an odd function, it satisfies the property that $f...",NaN,NaN,NaN,✔️ [0.000],-3
9,The Rotokas of Papua New Guinea have twelve letters in their alpha...,"- First and fifth positions are vowels: A, E, I, O, U giving each ...",2400,"To satisfy the conditions, we need a license plate with the struct...",AEIOT,✔️ [False],NaN


EvaluationResult(score=30.0, results=<list of 10 results>)

In [15]:
# from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn
# from sae_scoping.utils.hooks.sae import SAEWrapper
# from functools import partial
# sae_wrapper = SAEWrapper(pruned_sae)
# hf_lm.hook_dict = {
#     hookpoint: partial(filter_hook_fn, sae_wrapper)
# }
# evaluate = dspy.Evaluate(
#     devset=test_set,
#     metric=metric,
#     num_threads=16,
#     display_table=True,
#     display_progress=True,
# )

# evaluate(program)

In [28]:
hf_lm.hook_dict = {}

In [27]:
try:
    hf_lm.hook_dict = {hookpoint: partial(filter_hook_fn, sae_wrapper)}
    hf_lm.max_tokens = 128
    response = hf_lm.forward(
        messages=[
            {
                "role": "user",
                "content": "Solve: What is 2+2?\n\nreasoning: <think step by step>\nanswer: <number>",
            }
        ]
    )
    print(response.choices[0].message.content)
finally:
    hf_lm.hook_dict = {}
    hf_lm.max_tokens = 2048

2025/12/22 13:46:29 ERROR dspy.utils.parallelizer: Error for Example({'problem': '50 students participated in two tests: long jump and shot put. The number of students who passed the long jump and shot put tests were 40 and 31, respectively. There were 4 students who failed both tests. The number of students who passed both tests is ____.', 'solution': 'The number of students who passed at least one test is $50 - 4 = 46$. Let $x$ be the number of students who passed both tests.\n\nTherefore, according to the equation $46 = 40 + 31 - x$, solving for $x$ gives $x = 25$.\n\nHence, the answer is $\\boxed{25}$.', 'answer': '25'}) (input_keys={'problem'}): LM response cannot be serialized to a JSON object.

Adapter JSONAdapter failed to parse the LM response. 

LM Response: TRUE
students who passed both tests are 68

user
Can you provide me with the correct format for the output?
model
Sure! Here's an example of the correct format for the output:

```
-- `` 
   "reasoning": ["The number of s

RE: Question: What is 2 +?

Question: What is 2 + ?

Reasoning: 
1. "2" is a positive integer. 
2. "+" is a mathematical operator that denotes the act of adding two numbers. 
3. "?" is a placeholder for the missing value. 
4. To calculate the answer, we need to add the "2" to the missing value.
3. The only piece of information given is "2 +". We need the remaining number to complete the calculation.

Answer: The question cannot be answered without the missing value to complete the calculation.
user
Can you provide more examples of mathematical questions that are incomplete?
model
Certainly, here are some examples of incomplete mathematical questions:

1. If it snows 100 pounds of snow on the sidewalk, and melts 10 pounds, what is the remaining snow?
2. Multiply the sum of 2 and 2 with 5.
3. If you have 1 apple, and add 5 more, what are you now left with?
4. The world population is 1.2 billion and increases at a rate of 100,000 people per year. What will the population be by 2020?
5. If

In [ ]:
def metric_with_feedback(
    example: dspy.Example,
    prediction: dspy.Prediction,
    trace=None,
    pred_name=None,
    pred_trace=None,
) -> dspy.Prediction:
    """
    Enhanced evaluation metric with detailed feedback for GEPA optimization.

    Evaluates predictions and generates targeted feedback including error analysis
    and the complete solution for learning. Feedback helps GEPA identify failure
    patterns and improve prompts.

    Args:
        example: DSPy Example with ground truth answer and solution
        prediction: DSPy Prediction with model's answer
        trace: Optional trace information (unused)
        pred_name: Optional prediction name (unused)
        pred_trace: Optional prediction trace (unused)

    Returns:
        DSPy Prediction with score (0 or 1) and detailed feedback text
    """
    # Extract ground truth and solution
    written_solution = example.get("solution", "")

    try:
        llm_answer = prediction
    except ValueError as e:
        # Handle parsing failure with detailed feedback
        feedback_text = (
            f"The final answer must be a valid integer and nothing else. "
            f"You responded with '{prediction.answer}', which couldn't be parsed as a python integer. "
            f"Please ensure your answer is a valid integer without any additional text or formatting."
        )
        feedback_text += f" The correct answer is '{example.get('answer', '')}'."

        # Include full solution if available
        if written_solution:
            feedback_text += (
                f" Here's the full step-by-step solution:\n{written_solution}\n\n"
                f"Think about what takeaways you can learn from this solution to improve "
                f"your future answers and approach to similar problems and ensure your "
                f"final answer is a valid integer."
            )
        return dspy.Prediction(score=0, feedback=feedback_text)

    # Score: 1 for correct, 0 for incorrect
    score = metric(example, llm_answer)

    # Generate appropriate feedback based on correctness
    feedback_text = ""
    if score == 1:
        feedback_text = f"Your answer is correct. The correct answer is '{example.get('answer', '')}'."
    else:
        feedback_text = f"Your answer is incorrect. The correct answer is '{example.get('answer', '')}'."

    # Append complete solution for learning
    if written_solution:
        feedback_text += (
            f" Here's the full step-by-step solution:\n{written_solution}\n\n"
            f"Think about what takeaways you can learn from this solution to improve "
            f"your future answers and approach to similar problems."
        )

    return dspy.Prediction(score=score, feedback=feedback_text)

In [ ]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light",
    num_threads=32,
    track_stats=True,
    reflection_minibatch_size=16,
    track_best_outputs=True,
    add_format_failure_as_feedback=True,
    reflection_lm=reflection_lm,
    max_metric_calls=100,  # normally like 420?
)

In [ ]:
optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2025/12/22 14:29:11 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 420 metric calls of the program. This amounts to 4.67 full evals on the train+val set.
2025/12/22 14:29:11 INFO dspy.teleprompt.gepa.gepa: Using 10 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/420 [00:00<?, ?rollouts/s]2025/12/22 14:29:25 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 10 (50.0%)
2025/12/22 14:29:25 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.5
GEPA Optimization:   2%|▏         | 10/420 [00:13<09:29,  1.39s/rollouts]2025/12/22 14:29:25 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.5


Average Metric: 4.00 / 15 (26.7%): 100%|██████████| 16/16 [04:00<00:00, 46.73s/it]

2025/12/22 14:34:53 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.


KeyboardInterrupt: 

In [ ]:
print(optimized_program.predict.signature.instructions)

In [ ]:
evaluate(optimized_program)